In [2]:
# imports
import polars as pl
from pathlib import Path

In [14]:
FILE = Path('datasets/dataframe_final.pq')

In [15]:
df = pl.read_parquet(FILE)

In [17]:
COLUMNS = [
    'order_status',
    'order_approved_at',
    'review_score',
    'payment_value_sum',
    'count_payment_installments',
    'customer_state',
    'product_category_name',
    'product_photos_qty',
    'seller_state'
]

In [18]:
df = df.select(COLUMNS)

In [19]:
print(df)

shape: (99_441, 9)
┌──────────────┬──────────────┬──────────────┬─────────────┬───┬─────────────┬─────────────┬─────────────┬─────────────┐
│ order_status ┆ order_approv ┆ review_score ┆ payment_val ┆ … ┆ customer_st ┆ product_cat ┆ product_pho ┆ seller_stat │
│ ---          ┆ ed_at        ┆ ---          ┆ ue_sum      ┆   ┆ ate         ┆ egory_name  ┆ tos_qty     ┆ e           │
│ str          ┆ ---          ┆ i64          ┆ ---         ┆   ┆ ---         ┆ ---         ┆ ---         ┆ ---         │
│              ┆ str          ┆              ┆ f64         ┆   ┆ str         ┆ str         ┆ i64         ┆ str         │
╞══════════════╪══════════════╪══════════════╪═════════════╪═══╪═════════════╪═════════════╪═════════════╪═════════════╡
│ delivered    ┆ 2017-10-02   ┆ 4            ┆ 38.71       ┆ … ┆ SP          ┆ utilidades_ ┆ 4           ┆ SP          │
│              ┆ 11:07:15     ┆              ┆             ┆   ┆             ┆ domesticas  ┆             ┆             │
│ delivered  

In [23]:
# adjust data
df = df.with_columns(
    pl.col('order_approved_at')
      .str.to_datetime(format='%Y-%m-%d %H:%M:%S')
      .dt.date()
      .alias('order_approved_at')
)

In [24]:
df

order_status,order_approved_at,review_score,payment_value_sum,count_payment_installments,customer_state,product_category_name,product_photos_qty,seller_state
str,date,i64,f64,i64,str,str,i64,str
"""delivered""",2017-10-02,4,38.71,1,"""SP""","""utilidades_domesticas""",4,"""SP"""
"""delivered""",2018-07-26,4,141.46,1,"""BA""","""perfumaria""",1,"""SP"""
"""delivered""",2018-08-08,null,179.12,3,"""GO""","""automotivo""",1,"""SP"""
"""delivered""",2017-11-18,5,72.2,1,"""RN""","""pet_shop""",3,"""MG"""
"""delivered""",2018-02-13,null,28.62,1,"""SP""","""papelaria""",4,"""SP"""
…,…,…,…,…,…,…,…,…
"""delivered""",2017-03-09,null,85.08,3,"""SP""","""beleza_saude""",1,"""SP"""
"""delivered""",2018-02-06,4,195.0,3,"""SP""","""bebes""",4,"""SP"""
"""delivered""",2017-08-27,5,271.01,5,"""BA""","""eletrodomesticos_2""",2,"""SP"""


In [42]:
GRUPED_COLS = [
    'order_status',
    'order_approved_at',
    'review_score',
    'count_payment_installments',
    'customer_state',
    'product_category_name',
    'product_photos_qty',
    'seller_state'
]
df_grouped = df.group_by(GRUPED_COLS).agg(
    pl.col('payment_value_sum').mean().alias('total_value'),
    pl.len().alias('count')
)

In [43]:
df_grouped

order_status,order_approved_at,review_score,count_payment_installments,customer_state,product_category_name,product_photos_qty,seller_state,total_value,count
str,date,i64,i64,str,str,i64,str,f64,u32
"""delivered""",2017-03-07,null,1,"""SP""","""esporte_lazer""",1,"""PR""",42.42,1
"""delivered""",2017-07-08,null,1,"""SP""","""eletroportateis""",1,"""SP""",30.49,1
"""delivered""",2018-06-29,null,2,"""RJ""","""utilidades_domesticas""",6,"""SC""",81.34,1
"""delivered""",2018-03-23,1,3,"""RJ""","""moveis_decoracao""",1,"""SP""",193.22,1
"""delivered""",2017-12-07,null,1,"""MG""","""cool_stuff""",1,"""SP""",126.5,2
…,…,…,…,…,…,…,…,…,…
"""delivered""",2017-11-25,5,10,"""ES""","""cama_mesa_banho""",1,"""SP""",117.85,1
"""delivered""",2017-04-28,null,3,"""BA""","""telefonia""",2,"""SP""",34.78,1
"""delivered""",2017-11-07,null,10,"""PR""","""cama_mesa_banho""",1,"""PR""",104.26,1


In [45]:
df_grouped.write_excel('cubes/00_cubo.xlsx')

Com esse cubo e usando `pivot_tables` do `Excel` é possivel chegar nas visualizações desejadas.